

```

```



In [ ]:
# 1. Install Unsloth natively via their official pre-compiled GitHub wheel for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install essential dependencies without dependency-resolution lockups
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 3. Install utility libraries for Kaggle integration and dataset management
!pip install kagglehub datasets transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3tr66phg/unsloth_d3dc46a2da0a40cb9437d4c154fc6480
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3tr66phg/unsloth_d3dc46a2da0a40cb9437d4c154fc6480
  Resolved https://github.com/unslothai/unsloth.git to commit 94026fc8dc8e23423ec6bf232ba9ba6646cba21e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.7/842.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.2 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 0. DEEP VRAM FLUSH & MEMORY PURGE
# ==========================================
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 1. ENVIRONMENT SETUP & DATA IMPORT
# ==========================================
import kagglehub
import os
import json
from datasets import Dataset
from transformers import EarlyStoppingCallback, TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# ==========================================
# 2. ROBUST DATA LOADING
# ==========================================
def load_and_format_dataset(file_path):
    formatted_data = []
    line_count = 0
    skipped_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            records = json.load(f)
        else:
            records = []
            for line in f:
                line_count += 1
                cleaned_line = line.strip()
                if not cleaned_line: continue
                try:
                    records.append(json.loads(cleaned_line))
                except json.JSONDecodeError:
                    skipped_count += 1
                    continue

    for record in records:
        jd = record.get("job_description", "")
        questions = record.get("evaluation_questions", "")
        if not jd or not questions: continue

        messages = [
            {"role": "system", "content": "You are a professional HR assistant. Analyze the provided Job Description and generate categorized resume evaluation questions."},
            {"role": "user", "content": f"Generate evaluation questions for this job description:\n\n{jd}"},
            {"role": "assistant", "content": questions}
        ]
        formatted_data.append({"messages": messages})

    print(f"Successfully loaded {len(formatted_data)} records. (Skipped: {skipped_count} rows)")
    return Dataset.from_list(formatted_data)

In [ ]:
def apply_formatting_template(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

In [ ]:
dataset_dir = kagglehub.dataset_download('adhamashraf202200953/techncial-generated-job-descriptions')
print(f'Data source import complete. Directory path: {dataset_dir}')

all_files = []
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.endswith('.jsonl') or file.endswith('.json'):
            all_files.append(os.path.join(root, file))

INPUT_FILE = "/kaggle/input/techncial-generated-job-descriptions/phase2_questions.jsonl"
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "outputs/jd_to_questions_model"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

Using Colab cache for faster access to the 'techncial-generated-job-descriptions' dataset.
Data source import complete. Directory path: /kaggle/input/techncial-generated-job-descriptions


In [ ]:
raw_dataset = load_and_format_dataset(INPUT_FILE)

dataset_split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

template_type = "llama-3.1" if "llama" in MODEL_NAME.lower() else ("qwen-2.5" if "qwen" in MODEL_NAME.lower() else "phi-3.5")
tokenizer = get_chat_template(tokenizer, chat_template = template_type)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


train_dataset = train_dataset.map(apply_formatting_template, batched = True)
eval_dataset = eval_dataset.map(apply_formatting_template, batched = True)

Successfully loaded 5014 records. (Skipped: 1 rows)
==((====))==  Unsloth 2026.5.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Map:   0%|          | 0/4763 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

# ==========================================
# 4. TRAINING ARGUMENTS (THE MIDDLE GROUND)
# ==========================================
args = SFTConfig(
    dataset_text_field = "text",     # <--- MOVED HERE
    max_seq_length = MAX_SEQ_LENGTH, # <--- MOVED HERE
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    per_device_eval_batch_size = 1,
    eval_accumulation_steps = 1,
    warmup_steps = 10,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,
    save_total_limit = 2,
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    output_dir = "outputs",
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)]
)
trainer_stats = trainer.train()

best_checkpoint_path = trainer.state.best_model_checkpoint

if best_checkpoint_path is not None:
    model.from_pretrained(model, best_checkpoint_path)
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
else:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4763 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/251 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,763 | Num Epochs = 1 | Total steps = 596
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.619126,0.599109
200,0.560850,0.567317
300,0.537894,0.551341
400,0.528579,0.539810
500,0.532204,0.532622
596,0.520897,0.529075


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-596/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.mo

In [ ]:
# Source - https://stackoverflow.com/a/52555629
# Posted by Shashank Mishra
# Retrieved 2026-05-16, License - CC BY-SA 4.0

!zip -r "qwen_qag.zip" "/content/outputs"

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/jd_to_questions_model/ (stored 0%)
  adding: content/outputs/jd_to_questions_model/adapter_config.json (deflated 59%)
  adding: content/outputs/jd_to_questions_model/chat_template.jinja (deflated 71%)
  adding: content/outputs/jd_to_questions_model/README.md (deflated 65%)
  adding: content/outputs/jd_to_questions_model/tokenizer_config.json (deflated 89%)
  adding: content/outputs/jd_to_questions_model/adapter_model.safetensors (deflated 74%)
  adding: content/outputs/jd_to_questions_model/tokenizer.json (deflated 81%)
  adding: content/outputs/checkpoint-500/ (stored 0%)
  adding: content/outputs/checkpoint-500/training_args.bin (deflated 53%)
  adding: content/outputs/checkpoint-500/scaler.pt (deflated 64%)
  adding: content/outputs/checkpoint-500/rng_state.pth (deflated 26%)
  adding: content/outputs/checkpoint-500/adapter_config.json (deflated 59%)
  adding: content/outputs/checkpoint-500/chat_template.jinja (deflate

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs/jd_to_questions_model",
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an automated HR Screening Bot. Your job is to analyze the Job Description and generate "
            "objective verification check questions for an LLM to evaluate a Candidate's CV.\n\n"
            "CRITICAL RULES:\n"
            "1. DO NOT use second-person pronouns ('you', 'your', 'yours').\n"
            "2. DO NOT frame questions as a direct interview interview ('What is your experience...').\n"
            "3. Frame questions as a third-person verification query ('Does the CV show experience with...', "
            "'Is there evidence of the candidate managing...', 'Verify if the candidate has worked on...').\n"
            "4. Maintain strict markdown headers."
        )
    },
    {
        "role": "user",
        "content": f"Generate CV evaluation verification questions for this job description:\n\n{test_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.5,
    top_p = 0.9
)

print(tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True))

==((====))==  Unsloth 2026.5.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


### Verification Questions for CV

#### Kubernetes Cluster Administration Experience
1. **Does the CV show experience with Kubernetes cluster administration?**
2. **Are there examples of successful deployments or scaling operations using Kubernetes?**

#### Terraform for AWS Infrastructure Management
3. **Is there evidence of working with Terraform for AWS infrastructure management?**
4. **Can you provide specific projects where Terraform was used to manage AWS resources?**

#### GitLab CI/CD Pipeline Development
5. **Does the CV demonstrate experience in building GitLab CI/CD pipelines?**
6. **Are there any examples of CI/CD pipelines that have been successfully implemented and maintained?**

#### Additional Skills (Prometheus and Grafana)
7. **Is there evidence of experience with Prometheus and Grafana?**
8. **Have you managed monitoring systems like Prometheus and Grafana in previous roles?**
